<tabla align="centro">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visite el aprendizaje profundo del MIT</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab2/PT_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab2/PT_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png" height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</tabla>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT Introducción al aprendizaje profundo. Reservados todos los derechos.
# 
# Licenciado bajo la Licencia MIT. No puede utilizar este archivo excepto en cumplimiento
# con la Licencia. Uso y/o modificación de este código fuera del MIT Introducción
# al Deep Learning debe hacer referencia a:
# 
# © MIT Introducción al aprendizaje profundo
# http://intotodeeplearning.com
# 

# Laboratorio 2: Visión por Computador

# Parte 1: Clasificación de dígitos MNIST

En la primera parte de esta práctica de laboratorio, construiremos y entrenaremos una red neuronal convolucional (CNN) para clasificar dígitos escritos a mano del famoso conjunto de datos [MNIST](http://yann.lecun.com/exdb/mnist/). El conjunto de datos MNIST consta de 60.000 imágenes de entrenamiento y 10.000 imágenes de prueba. Nuestras clases son los dígitos 0-9.

Primero, descarguemos el repositorio del curso, instalemos las dependencias e importemos los paquetes relevantes que necesitaremos para esta práctica de laboratorio.

In [ ]:
# Importe PyTorch y otras bibliotecas relevantes
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchsummary import summary

# Paquete de introducción al aprendizaje profundo del MIT
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# otros paquetes
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm import tqdm

También instalaremos Comet. Si siguió las instrucciones del Laboratorio 1, debería tener configurada su cuenta Comet. Ingrese su clave API a continuación.

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ¡¡INGRESA TU CLAVE API AQUÍ!!
COMET_API_KEY = ""

# Comprobar que estamos usando una GPU, si no cambiar de tiempo de ejecución
# usando Runtime > Cambiar tipo de tiempo de ejecución > GPU
assert torch.cuda.is_available(), "Please enable GPU from runtime settings"
assert COMET_API_KEY != "", "Please insert your Comet API Key"

# Configurar GPU para computación
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# iniciar un primer experimento con cometas para la primera parte del laboratorio
comet_ml.init(project_name="6S191_lab2_part1_NN")
comet_model_1 = comet_ml.Experiment()

## 1.1 conjunto de datos MNIST

Descarguemos y carguemos el conjunto de datos y mostremos algunas muestras aleatorias del mismo:

In [ ]:
# Descargue y transforme el conjunto de datos MNIST
transform = transforms.Compose([
    # Convierta imágenes a tensores de PyTorch que también escalan datos de [0,255] a [0,1]
    transforms.ToTensor()
])

# Descargar conjuntos de datos de entrenamiento y prueba
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

El objeto del conjunto de datos MNIST en PyTorch no es un simple tensor o matriz. Es un conjunto de datos iterable que carga muestras (pares de imagen-etiqueta) de una en una o en lotes. En una sección posterior de esta práctica de laboratorio, definiremos un útil DataLoader para procesar los datos en lotes.

In [ ]:
image, label = train_dataset[0]
print(image.size())  # Para un tensor: antorcha.Tamaño([1, 28, 28])
print(label)  # Para una etiqueta: número entero (por ejemplo, 5)

Nuestro conjunto de capacitación se compone de imágenes en escala de grises de 28x28 de dígitos escritos a mano.

Visualicemos cómo lucen algunas de estas imágenes y sus correspondientes etiquetas de entrenamiento.

In [ ]:
plt.figure(figsize=(10,10))
random_inds = np.random.choice(60000,36)
for i in range(36):
    plt.subplot(6, 6, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    image_ind = random_inds[i]
    image, label = train_dataset[image_ind]
    plt.imshow(image.squeeze(), cmap=plt.cm.binary)
    plt.xlabel(label)
comet_model_1.log_figure(figure=plt)

## 1.2 Red neuronal para clasificación de dígitos escritos a mano

Primero construiremos una red neuronal simple que consta de dos capas completamente conectadas y la aplicaremos a la tarea de clasificación de dígitos. En última instancia, nuestra red generará una distribución de probabilidad entre las clases de 10 dígitos (0-9). Esta primera arquitectura que construiremos se muestra a continuación:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/mnist_2layers_arch.png "CNN Architecture for MNIST Classification")


### Arquitectura de red neuronal totalmente conectada
Para definir la arquitectura de esta primera red neuronal completamente conectada, usaremos una vez más los módulos `torch.nn`, definiendo el modelo usando [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html). Observe cómo usamos primero una capa [`nn.Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), que aplana la entrada para que pueda introducirse en el modelo.

En el siguiente bloque, definirá las capas completamente conectadas de esta red simple.

In [ ]:
def build_fc_model():
    fc_model = nn.Sequential(
        # Primero defina una capa Aplanar
        nn.Flatten(),

        # '''TODO: Definir la función de activación para la primera capa completamente conectada (Densa/Lineal).'''
        nn.Linear(28 * 28, 128),
        '''TODO'''

        '''TODO: Define the second Linear layer to output the classification probabilities'''
        )
    return fc_model

fc_model_sequential = build_fc_model()

A medida que avancemos en la siguiente parte, es posible que desee realizar cambios en la arquitectura definida anteriormente. **Tenga en cuenta que para actualizar el modelo más adelante, deberá volver a ejecutar la celda anterior para reinicializar el modelo.**

Demos un paso atrás y pensemos en la red que acabamos de crear. La primera capa de esta red, `nn.Flatten`, transforma el formato de las imágenes de una matriz 2D (28 x 28 píxeles) a una matriz 1D de 28 * 28 = 784 píxeles. Puede pensar en esta capa como desapilar filas de píxeles en la imagen y alinearlas. No hay parámetros aprendidos en esta capa; solo reformatea los datos.

Una vez aplanados los píxeles, la red consta de una secuencia de dos capas "nn.Linear". Estas son capas neuronales completamente conectadas. La primera capa `nn.Linear` tiene 128 nodos (o neuronas). La segunda (y última) capa (¡que ha definido!) debe devolver una serie de puntuaciones de probabilidad que suman 1. Cada nodo contiene una puntuación que indica la probabilidad de que la imagen actual pertenezca a una de las clases de dígitos escritos a mano.

¡Eso define nuestro modelo totalmente conectado!

### Adoptando subclases en PyTorch

Recuerde que en el Laboratorio 1 exploramos la creación de modelos más flexibles subclasificando [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). Esta técnica de definir modelos se usa más comúnmente en PyTorch. Practicaremos el uso de este enfoque de subclases para definir nuestros modelos para el resto del laboratorio.

In [ ]:
# Definir el modelo completamente conectado
class FullyConnectedModel(nn.Module):
    def __init__(self):
        super(FullyConnectedModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)

        # '''TODO: Definir la función de activación para la primera capa completamente conectada'''
        self.relu = # TODO

        # '''TODO: Definir la segunda capa lineal para generar las probabilidades de clasificación'''
        self.fc2 = # TODO

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)

        # '''TODO: Implementar el resto del paso hacia adelante del modelo usando las capas que has definido arriba'''
        '''TODO'''

        return x

fc_model = FullyConnectedModel().to(device) # enviar el modelo a la GPU

### Métricas del modelo y parámetros de entrenamiento

Antes de entrenar el modelo, necesitamos definir componentes que gobiernen su desempeño y guíen su proceso de aprendizaje. Estos incluyen la función de pérdida, el optimizador y las métricas de evaluación:

* *Función de pérdida*: define cómo medimos la precisión del modelo durante el entrenamiento. Como se explicó en la conferencia, durante el entrenamiento queremos minimizar esta función, lo que "dirigirá" el modelo en la dirección correcta.
* *Optimizador*: define cómo se actualiza el modelo en función de los datos que ve y su función de pérdida.
* *Métricas*: aquí podemos definir las métricas que queremos usar para monitorear los pasos de capacitación y prueba. En este ejemplo, definiremos y veremos la *precisión*, la fracción de las imágenes que están clasificadas correctamente.

Comenzaremos utilizando un optimizador de descenso de gradiente estocástico (SGD) inicializado con una tasa de aprendizaje de 0,1. Dado que estamos realizando una tarea de clasificación categórica, querremos utilizar [cross entropy loss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

Querrá experimentar tanto con la elección del optimizador como con la tasa de aprendizaje y evaluar cómo afectan la precisión del modelo entrenado.

In [ ]:
'''TODO: Experiment with different optimizers and learning rates. How do these affect
    the accuracy of the trained model? Which optimizers and/or learning rates yield
    the best performance?'''
# Definir función de pérdida y optimizador.
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(fc_model.parameters(), lr=0.1)

### Entrenar el modelo

Ahora estamos listos para entrenar nuestro modelo, lo que implicará introducir los datos de entrenamiento (`train_dataset`) en el modelo y luego pedirle que aprenda las asociaciones entre imágenes y etiquetas. También necesitaremos definir el tamaño del lote y la cantidad de épocas, o iteraciones sobre el conjunto de datos MNIST, que se usarán durante el entrenamiento. Este conjunto de datos consta de tuplas (imagen, etiqueta) a las que accederemos iterativamente en lotes.

En el laboratorio 1, vimos cómo podemos usar el método [`.backward()`](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html) para optimizar las pérdidas y entrenar modelos con descenso de gradiente estocástico. En esta sección, definiremos una función para entrenar el modelo usando `.backward()` y `optimizer.step()` para actualizar automáticamente los parámetros de nuestro modelo (pesos y sesgos) como vimos en el Laboratorio 1.

Recuerde, mencionamos en la Sección 1.1 que se puede acceder al conjunto de datos MNIST de forma iterativa en lotes. Aquí, definiremos un PyTorch [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) que nos permitirá hacer eso.

In [ ]:
# Cree DataLoaders para procesamiento por lotes
BATCH_SIZE = 64
trainset_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def train(model, dataloader, criterion, optimizer, epochs):
    model.train()  # Establecer el modelo en modo de entrenamiento.
    for epoch in range(epochs):
        total_loss = 0
        correct_pred = 0
        total_pred = 0

        for images, labels in trainset_loader:
            # Mueva los tensores a la GPU para que sean compatibles con el modelo
            images, labels = images.to(device), labels.to(device)

            # pase hacia adelante
            outputs = fc_model(images)

            # Borrar gradientes antes de realizar un pase hacia atrás
            optimizer.zero_grad()
            # Calcular la pérdida según las predicciones del modelo.
            loss = loss_function(outputs, labels)
            # Retropropagar y actualizar los parámetros del modelo.
            loss.backward()
            optimizer.step()

            # multiplique la pérdida por el total de números. de muestras en lote
            total_loss += loss.item()*images.size(0)

            # Calcular la precisión
            predicted = torch.argmax(outputs, dim=1)  # Obtener clase prevista
            correct_pred += (predicted == labels).sum().item()  # Contar predicciones correctas
            total_pred += labels.size(0) # Contar las predicciones totales

        # Calcular métricas
        total_epoch_loss = total_loss / total_pred
        epoch_accuracy = correct_pred / total_pred
        print(f"Epoch {epoch + 1}, Loss: {total_epoch_loss}, Accuracy: {epoch_accuracy:.4f}")


In [ ]:
# TODO: entrenar el modelo llamando a la función apropiadamente
EPOCHS = 5
train('''TODO''') # TODO

comet_model_1.end()

A medida que el modelo se entrena, se muestran las métricas de pérdida y precisión. Con cinco épocas y una tasa de aprendizaje de 0,01, este modelo completamente conectado debería lograr una precisión de aproximadamente 0,97 (o 97%) en los datos de entrenamiento.

### Evaluar la precisión en el conjunto de datos de prueba

Ahora que hemos entrenado el modelo, podemos pedirle que haga predicciones sobre un conjunto de pruebas que no haya visto antes. En este ejemplo, iterar sobre `testset_loader` nos permite acceder a nuestras imágenes y etiquetas de prueba. Y para evaluar la precisión, podemos verificar si las predicciones del modelo coinciden con las etiquetas de este cargador.

Como ahora hemos entrenado el modo, usaremos el estado de evaluación del modelo en el conjunto de datos de prueba.

In [ ]:
'''TODO: Use the model we have defined in its eval state to complete
and call the evaluate function, and calculate the accuracy of the model'''

def evaluate(model, dataloader, loss_function):
    # Evaluar el rendimiento del modelo en el conjunto de datos de prueba
    model.eval()
    test_loss = 0
    correct_pred = 0
    total_pred = 0
    # Deshabilitar los cálculos de gradiente cuando esté en modo de inferencia
    with torch.no_grad():
        for images, labels in testset_loader:
            # TODO: asegúrese de que la evaluación se realice en la GPU
            images, labels = # TODO

            # TODO: introducir las imágenes en el modelo y obtener las predicciones (pase hacia adelante)
            outputs = # TODO

            loss = loss_function(outputs, labels)

            # TODO: Calcular la pérdida de prueba
            test_loss += # TODO

           '''TODO: make a prediction and determine whether it is correct!'''
            # TODO: identifique el dígito con la predicción de mayor probabilidad para las imágenes en el conjunto de datos de prueba.
            predicted = # antorcha.argmax('''TODO''')

            # TODO: contar el número de predicciones correctas
            correct_pred += TODO

            # TODO: contar el número total de predicciones
            total_pred += TODO

    # Calcular la pérdida promedio y la precisión
    test_loss /= total_pred
    test_acc = correct_pred / total_pred
    return test_loss, test_acc

# TODO: ¡llame a la función de evaluación para evaluar el modelo entrenado!
test_loss, test_acc = # TODO

print('Precisión de la prueba:', test_acc)

Puede observar que la precisión del conjunto de datos de prueba es un poco menor que la precisión del conjunto de datos de entrenamiento. Esta brecha entre la precisión del entrenamiento y la precisión de las pruebas es un ejemplo de *sobreajuste*, cuando un modelo de aprendizaje automático funciona peor con datos nuevos que con sus datos de entrenamiento.

¿Cuál es la mayor precisión que puede lograr con este primer modelo totalmente conectado? Dado que la tarea de clasificación de dígitos escritos a mano es bastante sencilla, es posible que se pregunte cómo podemos hacerlo mejor...

![Deeper...](https://i.kym-cdn.com/photos/images/newsfeed/000/534/153/f87.jpg)

## 1.3 Red neuronal convolucional (CNN) para clasificación de dígitos escritos a mano

Como vimos en la conferencia, las redes neuronales convolucionales (CNN) son particularmente adecuadas para una variedad de tareas en visión por computadora y han logrado precisiones casi perfectas en el conjunto de datos MNIST. Ahora construiremos una CNN compuesta por dos capas convolucionales y capas de agrupación, seguidas de dos capas completamente conectadas y, en última instancia, generaremos una distribución de probabilidad entre las clases de 10 dígitos (0-9). La CNN que construiremos se muestra a continuación:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/convnet_fig.png "CNN Architecture for MNIST Classification")

### Definir el modelo CNN

Usaremos los mismos conjuntos de datos de entrenamiento y prueba que antes, y procederemos de manera similar a nuestra red completamente conectada para definir y entrenar nuestro nuevo modelo CNN. Para hacer esto, exploraremos dos capas que no hemos encontrado antes: puede usar [`nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) para definir capas convolucionales y [`nn.MaxPool2D`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) para definir las capas de agrupación. Utilice los parámetros que se muestran en la arquitectura de red anterior para definir estas capas y construir el modelo CNN. Puede decidir utilizar `nn.Sequential` o crear una subclase de `nn.Module` según sus preferencias.

In [ ]:
# ## CNN básica en PyTorch ###

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # TODO: Definir la primera capa convolucional
        self.conv1 = # TODO

        # TODO: Definir la primera capa de agrupación máxima
        self.pool1 = # TODO

        # TODO: Definir la segunda capa convolucional
        self.conv2 = # TODO

        # TODO: Definir la segunda capa de agrupación máxima
        self.pool2 = # TODO

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(36 * 5 * 5, 128)
        self.relu = nn.ReLU()

        # TODO: Definir la capa lineal que genera la clasificación
        # logits sobre etiquetas de clase. Recuerde que CrossEntropyLoss opera sobre logits.
        self.fc2 = # TODO


    def forward(self, x):
        # Primeras capas convolucionales y de agrupación
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)

        # '''TODO: Implementar el resto del paso hacia adelante del modelo usando las capas que has definido arriba'''
        # '''sugerencia: esto implicará otro conjunto de capas convolucionales/agrupadas y luego las capas lineales'''
        '''TODO'''

        return x

# Instanciar el modelo
cnn_model = CNN().to(device)
# Inicialice el modelo pasando algunos datos a través de
image, label = train_dataset[0]
image = image.to(device).unsqueeze(0)  # Agregar dimensión de lote → Forma: (1, 1, 28, 28)
output = cnn_model(image)
# Imprimir el resumen del modelo
print(cnn_model)

### Entrene y pruebe el modelo CNN

Anteriormente en el laboratorio, definimos una función de "entrenamiento". El cuerpo de la función es bastante útil porque nos permite tener control sobre el modelo de entrenamiento y registrar operaciones de diferenciación durante el entrenamiento calculando los gradientes usando `loss.backward()`. Quizás recuerde haber visto esto en el Laboratorio 1, Parte 1.

Usaremos este mismo marco para entrenar nuestro `cnn_model` usando un descenso de gradiente estocástico. Eres libre de implementar las siguientes partes con o sin las funciones de entrenamiento y evaluación que definimos anteriormente. Lo más importante es comprender cómo manipular los cuerpos de esas funciones para entrenar y probar modelos.

Como hicimos anteriormente, podemos definir la función de pérdida, el optimizador y calcular la precisión del modelo. Defina un optimizador y una tasa de aprendizaje de su elección. Siéntase libre de modificarlo como mejor le parezca para optimizar el rendimiento de su modelo.

In [ ]:
# Reconstruir el modelo CNN
cnn_model = CNN().to(device)

# Definir hiperparámetros
batch_size = 64
epochs = 7
optimizer = optim.SGD(cnn_model.parameters(), lr=1e-2)

# TODO: crear una instancia de la función de pérdida de entropía cruzada
loss_function = # TODO

# Redefinir el cargador de trenes con un nuevo parámetro de tamaño de lote (modificarlo como mejor le parezca si se optimiza)
trainset_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
testset_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
loss_history = mdl.util.LossHistory(smoothing_factor=0.95) # para registrar la evolución de la pérdida
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss', scale='semilogy')

# Inicializar nuevo experimento con cometas
comet_ml.init(project_name="6.s191lab2_part1_CNN")
comet_model_2 = comet_ml.Experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # claro si existe

# ¡Bucle de entrenamiento!
cnn_model.train()

for epoch in range(epochs):
    total_loss = 0
    correct_pred = 0
    total_pred = 0

    # Primero tome un lote de datos de entrenamiento que nuestro cargador de datos devuelve como un tensor.
    for idx, (images, labels) in enumerate(tqdm(trainset_loader)):
        images, labels = images.to(device), labels.to(device)

        # pase hacia adelante
        # TODO: introducir las imágenes en el modelo y obtener las predicciones.
        logits = # TODO

        # TODO: calcular la pérdida de entropía cruzada categórica utilizando los logits predichos
        loss = # TODO

        # Obtenga la pérdida y regístrela en el cometa y en el registro loss_history
        loss_value = loss.item()
        comet_model_2.log_metric("loss", loss_value, step=idx)
        loss_history.append(loss_value) # agregar la pérdida al registro loss_history
        plotter.plot(loss_history.get())

        # Propagación hacia atrás/paso hacia atrás
        '''TODO: Compute gradients for all model parameters and propagate backwads
            to update model parameters. remember to reset your optimizer!'''
        # TODO: restablecer el optimizador
        # TODO: compute gradients
        # TODO: actualizar los parámetros del modelo

        # Obtenga las métricas de predicción y recuento
        predicted = torch.argmax(logits, dim=1)
        correct_pred += (predicted == labels).sum().item()
        total_pred += labels.size(0)

    # Calcular métricas
    total_epoch_loss = total_loss / total_pred
    epoch_accuracy = correct_pred / total_pred
    print(f"Epoch {epoch + 1}, Loss: {total_epoch_loss}, Accuracy: {epoch_accuracy:.4f}")

comet_model_2.log_figure(figure=plt)

### Evaluar el modelo CNN

Ahora que hemos entrenado el modelo, vamos a evaluarlo en el conjunto de datos de prueba.

In [ ]:
'''TODO: Evaluate the CNN model!'''
test_loss, test_acc = evaluate('''TODO''')

print('Precisión de la prueba:', test_acc)

¿Cuál es la precisión más alta que puede lograr utilizando el modelo CNN y cómo se compara la precisión del modelo CNN con la precisión de la red simple completamente conectada? ¿Qué optimizadores y tasas de aprendizaje parecen óptimos para entrenar el modelo CNN?

No dude en hacer clic en los enlaces de Comet para investigar las curvas de entrenamiento/precisión de su modelo.

### Haz predicciones con el modelo CNN

Con el modelo entrenado, podemos usarlo para hacer predicciones sobre algunas imágenes.

In [ ]:
test_image, test_label = test_dataset[0]
test_image = test_image.to(device).unsqueeze(0)

# poner el modelo en modo de evaluación (inferencia)
cnn_model.eval()
predictions_test_image = cnn_model(test_image)

Con esta llamada a función, el modelo ha predicho la etiqueta de la primera imagen en el conjunto de prueba. Echemos un vistazo a la predicción:

In [ ]:
print(predictions_test_image)

Como puede ver, una predicción es una matriz de 10 números. Recuerde que el resultado de nuestro modelo es una distribución entre las clases de 10 dígitos. Por lo tanto, estos números describen la probabilidad predicha del modelo de que la imagen corresponda a cada uno de los 10 dígitos diferentes.

Veamos el dígito que tiene la mayor probabilidad de aparecer en la primera imagen del conjunto de datos de prueba:

In [ ]:
'''TODO: identify the digit with the highest likelihood prediction for the first
    image in the test dataset. '''
predictions_value = predictions_test_image.cpu().detach().numpy() # .cpu() para copiar el tensor a la memoria primero
prediction = # TODO
print(prediction)

Entonces, el modelo está más seguro de que esta imagen es un "???". Podemos comprobar la etiqueta de prueba (recuerde, esta es la verdadera identidad del dígito) para ver si esta predicción es correcta:

In [ ]:
print("La etiqueta de este dígito es:", test_label)
plt.imshow(test_image[0,0,:,:].cpu(), cmap=plt.cm.binary)
comet_model_2.log_figure(figure=plt)

¡Es! Visualicemos los resultados de la clasificación en el conjunto de datos MNIST. Trazaremos imágenes del conjunto de datos de prueba junto con su etiqueta predicha, así como un histograma que proporciona las probabilidades de predicción para cada uno de los dígitos.

Recuerde que en PyTorch generalmente se accede al conjunto de datos MNIST mediante un DataLoader para recorrer el conjunto de pruebas en lotes más pequeños y manejables. Al agregar las predicciones, etiquetas de prueba e imágenes de prueba de cada lote, primero acumularemos gradualmente todos los datos necesarios para la visualización en variables singulares para observar las predicciones de nuestro modelo.

In [ ]:
# Inicializar variables para almacenar todos los datos.
all_predictions = []
all_labels = []
all_images = []

# Conjunto de pruebas de proceso en lotes
with torch.no_grad():
    for images, labels in testset_loader:
        outputs = cnn_model(images)

        # Aplique softmax para obtener probabilidades de los logits predichos
        probabilities = torch.nn.functional.softmax(outputs, dim=1)

        # Obtener clases previstas
        predicted = torch.argmax(probabilities, dim=1)

        all_predictions.append(probabilities)
        all_labels.append(labels)
        all_images.append(images)

all_predictions = torch.cat(all_predictions)  # Forma: (total_muestras, num_clases)
all_labels = torch.cat(all_labels)            # Forma: (total_muestras,)
all_images = torch.cat(all_images)            # Forma: (total_muestras, 1, 28, 28)

# Convierta tensores a NumPy para compatibilidad con funciones de trazado
predictions = all_predictions.cpu().numpy()  # Forma: (total_muestras, num_clases)
test_labels = all_labels.cpu().numpy()       # Forma: (total_muestras,)
test_images = all_images.cpu().numpy()       # Forma: (total_muestras, 1, 28, 28)

In [ ]:
# @title ¡Cambie el control deslizante para ver las predicciones del modelo! {ejecutar: "automático" }

image_index = 79 # @param {tipo:"control deslizante", min:0, max:100, paso:1}
plt.subplot(1,2,1)
mdl.lab2.plot_image_prediction(image_index, predictions, test_labels, test_images)
plt.subplot(1,2,2)
mdl.lab2.plot_value_prediction(image_index, predictions, test_labels)
comet_model_2.log_figure(figure=plt)

También podemos trazar varias imágenes junto con sus predicciones, donde las etiquetas de predicción correcta son azules y las etiquetas de predicción incorrecta son grises. El número proporciona el porcentaje de confianza (sobre 100) de la etiqueta prevista. Tenga en cuenta que el modelo puede tener mucha confianza en una predicción incorrecta.

In [ ]:
# Traza las primeras X imágenes de prueba, su etiqueta predicha y la etiqueta verdadera
# Colorea las predicciones correctas en azul y las incorrectas en rojo.
num_rows = 5
num_cols = 4
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  mdl.lab2.plot_image_prediction(i, predictions, test_labels, test_images)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  mdl.lab2.plot_value_prediction(i, predictions, test_labels)
comet_model_2.log_figure(figure=plt)
comet_model_2.end()

## 1.5 Conclusión
En esta parte del laboratorio, tuvo la oportunidad de jugar con diferentes clasificadores MNIST con diferentes arquitecturas (solo capas completamente conectadas, CNN) y experimentar cómo los diferentes hiperparámetros afectan la precisión (tasa de aprendizaje, etc.). La siguiente parte del laboratorio explora otra aplicación de las CNN, la detección facial y algunos inconvenientes de los sistemas de IA en aplicaciones del mundo real, como problemas de sesgo.